# 🔍 Inference: Base Model vs. Fine-Tuned Coding Assistant

*Prepared by **Jude Teves***.<br>

---

Runs the same 4 test prompts through:
1. **Base model** — `Llama-3.2-1B-Instruct` with no fine-tuning
2. **Fine-tuned model** — with the LoRA adapter from `01_train.ipynb`

The fine-tuned model includes a tool execution loop: when it outputs a `<tool_call>` block, the corresponding Python function runs and the result is fed back for a final answer.

For reference, here are the 4 test prompts:
```python
test_prompts = [
    ('Code question (direct)',    'How do I reverse a list in Python?'),
    ('Error lookup (tool)',       "I'm getting: ModuleNotFoundError: No module named 'pandas'"),
    ('Library suggestion (tool)', 'What library should I use to parse PDFs in Python?'),
    ('Off-topic (refusal)',       'What should I have for lunch today?'),
]
```

## 1. Install

In [1]:
%%capture
!pip install unsloth
!pip install trl==0.15.2 transformers==4.51.3

## 2. Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/SLMDev/'
ADAPTER_PATH = DRIVE_PATH + 'coding_assistant_lora'
print(f'Adapter will be loaded from: {ADAPTER_PATH}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Adapter will be loaded from: /content/drive/MyDrive/SLMDev/coding_assistant_lora


## 3. Tool Implementations

Five tools the fine-tuned model can call. `run_snippet` and `fetch_docs` actually execute; the others use lookup dicts. All return plain strings fed back into the conversation.

In [3]:
import json, re, sys
from io import StringIO

# ── Tool functions ──────────────────────────────────────────────

def run_snippet(language, code):
    if language.lower() != 'python':
        return 'Only Python execution is supported in this demo.'
    _stdout = sys.stdout          # save reference
    buf = StringIO()
    sys.stdout = buf
    try:
        exec(code, {})
    except Exception as e:
        sys.stdout = _stdout      # restore immediately on error
        return f'Error: {e}'
    sys.stdout = _stdout          # restore on success
    return buf.getvalue().strip() or '(no output)'

def fetch_docs(package, function):
    import pydoc
    try:
        obj = eval(f'{package}.{function}')
        return pydoc.render_doc(obj, renderer=pydoc.plaintext)[:600]
    except Exception as e:
        return f'Could not fetch docs: {e}'

def lookup_error(error_message):
    db = {
        'ModuleNotFoundError': 'The package is not installed. Run: pip install <package_name>',
        'TypeError':           'Wrong type passed to a function. Check argument types.',
        'KeyError':            'Dict key does not exist. Use .get() or check with `in` first.',
        'IndexError':          'List index out of range. Check len() before accessing.',
        'AttributeError':      'Object does not have that attribute. Check spelling or object type.',
        'RecursionError':      'Infinite recursion. Ensure your base case is reachable.',
        'ValueError':          'Correct type but invalid value. Check input constraints.',
        'NameError':           'Variable or function not defined. Check spelling and scope.',
    }
    for key, explanation in db.items():
        if key in error_message:
            return f'[{key}] {explanation}'
    return 'Unknown error. Examine the full stack trace for context.'

def suggest_library(task_description):
    td = task_description.lower()
    db = [
        (['pdf'],                   'pdfplumber — better than pypdf for structured extraction'),
        (['http', 'request', 'api'],'httpx — supports async; requests for simple sync use'),
        (['data', 'csv', 'table'],  'pandas for tabular data; polars if you need speed'),
        (['scraping', 'scrape'],    'beautifulsoup4 + requests; playwright for JS-heavy sites'),
        (['cli', 'command line'],   'typer — auto-complete and type hints built in'),
        (['valid', 'schema'],       'pydantic — type-safe validation with clear error messages'),
        (['test'],                  'pytest — simple, powerful, great plugin ecosystem'),
        (['async', 'concurrent'],   'asyncio + httpx or aiohttp for async I/O'),
    ]
    for keywords, suggestion in db:
        if any(k in td for k in keywords):
            return suggestion
    return 'No strong match — describe the task more specifically.'

def search_codebase(query):
    # Dummy — replace with real grep/glob if files are available
    fake_results = {
        'auth':     'src/auth/middleware.py (line 42), src/auth/tokens.py (line 18)',
        'database': 'src/db/connection.py (line 5), src/db/models.py (line 1)',
        'config':   'src/config.py (line 1), .env.example',
    }
    for key, result in fake_results.items():
        if key in query.lower():
            return f'Found matches: {result}'
    return f'No results found for: {query}'

def code_and_run(task_description):
    """Generate Python from a natural language description, then execute it."""
    # First pass: generate code only
    code_messages = [
        {'role': 'system', 'content': 'You are a Python code generator. Return ONLY raw Python code. No explanation, no markdown fences.'},
        {'role': 'user',   'content': task_description},
    ]
    # Use ft_model to generate the code
    inputs = ft_tokenizer.apply_chat_template(
        code_messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')
    import torch
    with torch.no_grad():
        out = ft_model.generate(input_ids=inputs, max_new_tokens=200, use_cache=True, do_sample=False)
    generated_code = ft_tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()
    # Strip markdown fences if model adds them
    generated_code = re.sub(r'^```python\n?', '', generated_code)
    generated_code = re.sub(r'\n?```$', '', generated_code).strip()
    print(f'  [generated code]\n{generated_code}\n')
    return run_snippet('python', generated_code)

TOOLS = {
    'run_snippet':     run_snippet,
    'fetch_docs':      fetch_docs,
    'lookup_error':    lookup_error,
    'suggest_library': suggest_library,
    'search_codebase': search_codebase,
    'code_and_run':    code_and_run,
}

def execute_tool(tool_call_dict):
    name = tool_call_dict.get('name')
    args = tool_call_dict.get('arguments', {})
    fn = TOOLS.get(name)
    if not fn:
        return f'Unknown tool: {name}'
    return str(fn(**args))

print('Tools ready:', list(TOOLS.keys()))

Tools ready: ['run_snippet', 'fetch_docs', 'lookup_error', 'suggest_library', 'search_codebase', 'code_and_run']


## 4. Inference Helpers

Two functions:
- `run_base(prompt)` — single-shot generation, no tool loop
- `run_finetuned(prompt)` — checks for `<tool_call>`, executes it, feeds result back for final answer

In [4]:
import torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

SYSTEM = (
    "You are a coding assistant. You only answer programming and software development questions. "
    "For anything unrelated to code or software, politely decline. "
    "When you need to look something up, run code, or find a library, use the available tools "
    "by outputting a <tool_call> JSON block."
)

def generate(model, tokenizer, messages, max_new_tokens=256):
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')
    with torch.no_grad():
        out = model.generate(
            input_ids=inputs,
            max_new_tokens=max_new_tokens,
            use_cache=True,
            temperature=1.0,
            do_sample=False,
        )
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()

def run_base(model, tokenizer, user_msg):
    messages = [{'role': 'user', 'content': user_msg}]
    return generate(model, tokenizer, messages)

def run_finetuned(model, tokenizer, user_msg):
    messages = [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user',   'content': user_msg},
    ]
    response = generate(model, tokenizer, messages)

    # Check for tool call
    match = re.search(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', response, re.DOTALL)
    if match:
        tool_dict = json.loads(match.group(1))
        tool_result = execute_tool(tool_dict)
        print(f'  [tool: {tool_dict["name"]}] → {tool_result[:80]}')

        messages.append({'role': 'assistant', 'content': response})
        messages.append({'role': 'tool',      'content': tool_result})
        response = generate(model, tokenizer, messages)

    return response

print('Helpers defined.')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/unsloth_zoo/__init__.py:405: UserWarning: Unsloth fused-forward install skipped: requires transformers >= 4.56.0.
  _install_fused_forward()


🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not find `steps_per_generation` in grpo_trainer
Unsloth: Could not find `generation_batch_size` in grpo_trainer
Helpers defined.


## 5. Load Base Model & Run Test Prompts

In [5]:
# Load base model (no adapter)
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(base_model)
base_tokenizer = get_chat_template(base_tokenizer, chat_template='llama-3.1')
print('Base model loaded.')

==((====))==  Unsloth 2026.6.8: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Base model loaded.


In [6]:
test_prompts = [
    ('Code question (direct)',      'How do I reverse a list in Python?'),
    ('Error lookup (tool)',         "I'm getting: ModuleNotFoundError: No module named 'pandas'"),
    ('Library suggestion (tool)',   'What library should I use to parse PDFs in Python?'),
    ('Off-topic (refusal)',         'What should I have for lunch today?'),
]

base_results = {}
for label, prompt in test_prompts:
    print(f'\n{'='*60}')
    print(f'[BASE] {label}')
    print(f'User: {prompt}')
    answer = run_base(base_model, base_tokenizer, prompt)
    base_results[label] = answer
    print(f'Response: {answer}')

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



[BASE] Code question (direct)
User: How do I reverse a list in Python?
Response: Reversing a List in Python

You can reverse a list in Python using the following methods:

### Method 1: Using Slicing

You can use slicing to reverse a list. The syntax is as follows:

```python
list_name = [1, 2, 3, 4, 5]
reversed_list = list_name[::-1]
print(reversed_list)  # Output: [5, 4, 3, 2, 1]
```

In this method, `list_name[::-1]` means start at the end of the list and end at position 0, move with the step -1.

### Method 2: Using the `reversed` Function

You can also use the `reversed` function to reverse a list. The syntax is as follows:

```python
list_name = [1, 2, 3, 4, 5]
reversed_list = list(reversed(list_name))
print(reversed_list)  # Output: [5, 4, 3, 2, 1]
```

### Method 3: Using a Loop

You can also reverse a list using a loop. The syntax is as follows:

```python
list

[BASE] Error lookup (tool)
User: I'm getting: ModuleNotFoundError: No module named 'pandas'
Response: The `ModuleNo

## 6. Load Fine-Tuned Model & Run Same Prompts

In [7]:
# Load fine-tuned model with LoRA adapter from Drive
ft_model, ft_tokenizer = FastLanguageModel.from_pretrained(
    model_name = ADAPTER_PATH,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(ft_model)
ft_tokenizer = get_chat_template(ft_tokenizer, chat_template='llama-3.1')
print('Fine-tuned model loaded.')

==((====))==  Unsloth 2026.6.8: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2026.6.8 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


Fine-tuned model loaded.


In [8]:
ft_results = {}
for label, prompt in test_prompts:
    print(f'\n{'='*60}')
    print(f'[FINE-TUNED] {label}')
    print(f'User: {prompt}')
    answer = run_finetuned(ft_model, ft_tokenizer, prompt)
    ft_results[label] = answer
    print(f'Response: {answer}')


[FINE-TUNED] Code question (direct)
User: How do I reverse a list in Python?
Response: Use slicing: `my_list[::-1]` — returns a new reversed list. For in-place reversal: `my_list.reverse()`. Use slicing when you need the original untouched.

[FINE-TUNED] Error lookup (tool)
User: I'm getting: ModuleNotFoundError: No module named 'pandas'
  [tool: lookup_error] → [ModuleNotFoundError] The package is not installed. Run: pip install <package_na
Response: Let me install pandas if you haven't already.

<tool_call>
{"name": "install_package", "arguments": {"package_name": "pandas"}}
</tool_call>

[FINE-TUNED] Library suggestion (tool)
User: What library should I use to parse PDFs in Python?
  [tool: suggest_library] → pdfplumber — better than pypdf for structured extraction
Response: Let me look that up.

<tool_call>
{"name": "suggest_library", "arguments": {"task_description": "parse PDF in Python"}}
</tool_call>

[FINE-TUNED] Off-topic (refusal)
User: What should I have for lunch today?
R

## 7. Side-by-Side Comparison

Same prompts, same base model weights — only the fine-tuning changes the behavior.

In [9]:
import pandas as pd

rows = []
for label, prompt in test_prompts:
    rows.append({
        'Category':   label,
        'Prompt':     prompt[:60] + '...' if len(prompt) > 60 else prompt,
        'Base Model': base_results[label][:120] + '...' if len(base_results[label]) > 120 else base_results[label],
        'Fine-Tuned': ft_results[label][:120]  + '...' if len(ft_results[label])  > 120 else ft_results[label],
    })

df = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 120)
df


,Category,Prompt,Base Model,Fine-Tuned
0,Code question (direct),How do I reverse a list in Python?,Reversing a List in Python\n==========================\n\nYou can reverse a list in Python using the following metho...,Use slicing: `my_list[::-1]` — returns a new reversed list. For in-place reversal: `my_list.reverse()`. Use slicing ...
1,Error lookup (tool),I'm getting: ModuleNotFoundError: No module named 'pandas',"The `ModuleNotFoundError: No module named 'pandas'` error occurs when Python is unable to find the pandas library, w...","Let me install pandas if you haven't already.\n\n<tool_call>\n{""name"": ""install_package"", ""arguments"": {""package_nam..."
2,Library suggestion (tool),What library should I use to parse PDFs in Python?,There are several libraries you can use to parse PDFs in Python. Here are some popular ones:\n\n1. **PyPDF2**: This ...,"Let me look that up.\n\n<tool_call>\n{""name"": ""suggest_library"", ""arguments"": {""task_description"": ""parse PDF in Pyt..."
3,Off-topic (refusal),What should I have for lunch today?,There are so many options to choose from. Here are a few ideas to get you started:\n\n1. **Classic combos**: Grilled...,I'm strictly a coding assistant — food is a bit outside my scope. Anything code-related I can help you with?


## 8. Live Demo — Natural Language → Code → Execute

Try any natural language coding ask. The fine-tuned model calls `code_and_run`, generates Python, executes it, and returns the result.

In [13]:
# ── Change this prompt to demo different tasks ──────────────────
user_input = 'Show me the first 10 fibonacci numbers.'
# user_input = 'Calculate the sum of all even numbers from 1 to 100 and print it.'
# user_input = 'Print a multiplication table for 5.'
# user_input = 'What should I have for lunch?'  # should refuse

print(f'User: {user_input}\n')
print('─' * 50)
result = run_finetuned(ft_model, ft_tokenizer, user_input)
print(f'\nFinal answer: {result}')


User: Show me the first 10 fibonacci numbers.

──────────────────────────────────────────────────
  [generated code]
for i in range(10):
    if i < 2:
        print(i)
    else:
        print(f'{i-2} + {i-1}')

  [tool: code_and_run] → 0
1
0 + 1
1 + 2
2 + 3
3 + 4
4 + 5
5 + 6
6 + 7
7 + 8

Final answer: Let me do that again.

<tool_call>
{"name": "code_and_run", "arguments": {"task_description": "print the first 10 fibonacci numbers"}}
</tool_call>


In [12]:
# ── Change this prompt to demo different tasks ──────────────────
# user_input = 'Show me the first 10 fibonacci numbers.'
user_input = 'Calculate the sum of all even numbers from 1 to 100 and print it.'
# user_input = 'Print a multiplication table for 5.'
# user_input = 'What should I have for lunch?'  # should refuse

print(f'User: {user_input}\n')
print('─' * 50)
result = run_finetuned(ft_model, ft_tokenizer, user_input)
print(f'\nFinal answer: {result}')


User: Calculate the sum of all even numbers from 1 to 100 and print it

──────────────────────────────────────────────────
  [generated code]
sum_of_evens = sum(i for i in range(1, 101) if i % 2 == 0)
print(sum_of_evens)

  [tool: code_and_run] → 2550

Final answer: Running that now.


In [14]:
# ── Change this prompt to demo different tasks ──────────────────
# user_input = 'Show me the first 10 fibonacci numbers.'
# user_input = 'Calculate the sum of all even numbers from 1 to 100 and print it.'
user_input = 'Print a multiplication table for 5.'
# user_input = 'What should I have for lunch?'  # should refuse

print(f'User: {user_input}\n')
print('─' * 50)
result = run_finetuned(ft_model, ft_tokenizer, user_input)
print(f'\nFinal answer: {result}')

User: Print a multiplication table for 5.

──────────────────────────────────────────────────
  [generated code]
for i in range(1, 6):
    print(f'{i} × 5 = {i * 5}')

  [tool: code_and_run] → 1 × 5 = 5
2 × 5 = 10
3 × 5 = 15
4 × 5 = 20
5 × 5 = 25

Final answer: Let me know if you need anything else.


In [15]:
# ── Change this prompt to demo different tasks ──────────────────
# user_input = 'Show me the first 10 fibonacci numbers.'
# user_input = 'Calculate the sum of all even numbers from 1 to 100 and print it.'
# user_input = 'Print a multiplication table for 5.'
user_input = 'What should I have for lunch?'  # should refuse

print(f'User: {user_input}\n')
print('─' * 50)
result = run_finetuned(ft_model, ft_tokenizer, user_input)
print(f'\nFinal answer: {result}')


User: What should I have for lunch?

──────────────────────────────────────────────────

Final answer: I'm strictly a coding assistant — food is a bit outside my scope. Anything code-related I can help you with?
